# La factura de Londres

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelvo yo en vivo. Ustedes lo siguen, lo corren y me interrumpen. Las apuestas y las respuestas van en la **hoja de bitácora** impresa. Al final le toman una foto y la adjuntan a la tarea de Classroom.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Trabajar con un dato que no cabe en memoria: leerlo por pedazos, resumir cada pedazo, juntar los resúmenes sin equivocarse y dejar cada versión del dato con su huella. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 2 (seleccionar y preparar datos), 3 (flujo reproducible y trazable), 4 (implementar y evaluar un modelo) y 5 (comunicar a quien no es técnico). |
| **Uso de IA** | Pueden pedirle a Claude que explique una línea o que escriba una gráfica. Deben poder explicar sin ayuda por qué el archivo no se lee de un golpe, qué resúmenes se pueden juntar entre pedazos, qué es la huella de una tabla y por qué un resultado sin versión del dato no se puede reproducir. |
| **Entregable** | La foto de la hoja de bitácora llena, en la tarea de Classroom. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path.cwd() / "taller-cd-ml-2026-2"] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

PALETA = ["#1D4E89", "#D7263D", "#1B998B", "#6C4AB6", "#5BC0EB", "#F49097", "#9AA5B1"]      # colores del curso
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": plt.cycler(color=PALETA), "axes.titleweight": "bold", "figure.dpi": 110})
print("Herramientas listas.")

## 1. La pregunta

Entre 2011 y 2014, la empresa UK Power Networks instaló medidores inteligentes en 5.567 hogares de Londres y guardó el consumo de cada uno cada media hora. El resultado son 168 archivos con un millón de filas cada uno.

Imagínense una comercializadora de energía que atiende esos hogares y compra la energía un día antes.

1. **Predecir.** ¿Cuánto consume mañana un hogar promedio?
2. **Explicar.** ¿Qué mueve el consumo: la estación, el día de la semana, el tipo de hogar?
3. **Decidir.** ¿Con qué pronóstico se le avisa al cliente que mañana será un día de consumo alto?

**Antes de seguir, escriban en la hoja sus apuestas A1 a A4.** Se escriben con esfero y no se corrigen.

In [ ]:
from taller import londres
from taller.fuentes_clase03 import cuales, TARIFA_PLANA

ARCHIVOS = 4      # cuántos de los 168 archivos oficiales leemos en vivo. Al final lo cambiarán por 8

## 2. Un dato que no se baja entero

La fuente oficial es un archivo ZIP. Un ZIP guarda al final un índice con el nombre, el tamaño y la posición de cada archivo que lleva adentro. El servidor acepta que le pidan un rango de bytes. Así que leemos el índice y después solo los archivos que queramos.

In [ ]:
paquete, internos, remoto = londres.abrir()
print(f"Archivos en el ZIP: {len(internos)}")
print(f"Tamaño comprimido: {sum(i.compress_size for i in internos) / 1e9:.2f} GB | sin comprimir: {sum(i.file_size for i in internos) / 1e9:.1f} GB")
print(f"Para leer el índice bajamos {remoto.bytes_bajados / 1e6:.2f} MB")

### Un pedazo, mirado con calma

Antes de procesar nada conviene abrir un solo archivo y medirlo. Comparen la salida con sus apuestas A1 y A2.

In [ ]:
pedazo = londres.leer_pedazo(paquete, internos, 0)
print(f"Filas: {len(pedazo):,}".replace(",", "."))
print(f"Hogares distintos: {pedazo['LCLid'].nunique()}")
print(f"En disco: {internos[0].file_size / 1e6:.0f} MB | en memoria: {pedazo.memory_usage(deep=True).sum() / 1e6:.0f} MB")
pedazo.head(3)

El archivo está ordenado por hogar. Por eso un millón de filas son unas pocas decenas de hogares, cada uno con dos años de lecturas. Para generalizar importa el número de hogares, y este pedazo tiene pocos.

### Tres trampas del pedazo

In [ ]:
columna = pedazo.columns[3]
print("Nombre de la cuarta columna, con comillas:", repr(columna))
print("Lecturas que dicen Null:", int((pedazo[columna].str.strip() == "Null").sum()))
print("Filas repetidas idénticas:", int(pedazo.duplicated().sum()))

El nombre de la columna termina en un espacio. Algunas lecturas dicen `Null`, que es texto, y por eso la columna no se puede sumar. Y hay filas repetidas, que contarían doble.

## 3. Resumir cada pedazo y juntar los resúmenes

La comercializadora no necesita las lecturas de media hora. Necesita el consumo de cada hogar en cada día. Entonces cada pedazo se resume apenas se lee, y lo que se guarda es el resumen.

Hay una regla para juntar resúmenes: **se guardan sumas y cuentas, nunca promedios**. Un hogar puede quedar partido entre dos archivos, y el promedio de dos promedios no es el promedio del total:

$$\bar x=\frac{\sum_k s_k}{\sum_k n_k}\qquad\text{en general no es}\qquad\frac{1}{K}\sum_k \bar x_k .$$

In [ ]:
def resumir(crudo):
    """De lecturas de media hora a una fila por hogar y por día."""
    crudo = crudo.rename(columns=lambda nombre: nombre.strip()).drop_duplicates()      # trampas 1 y 3
    crudo["kwh"] = pd.to_numeric(crudo["KWH/hh (per half hour)"], errors="coerce")    # trampa 2: "Null" queda como faltante
    crudo["fecha"] = crudo["DateTime"].str[:10]
    return (crudo.groupby(["LCLid", "stdorToU", "fecha"])["kwh"].agg(kwh="sum", lecturas="count").reset_index()
            .rename(columns={"LCLid": "hogar", "stdorToU": "tarifa"}))

resumenes = {0: resumir(pedazo)}
for numero in cuales(ARCHIVOS)[1:]:
    resumenes[numero] = resumir(londres.leer_pedazo(paquete, internos, numero))
    print(f"Archivo {numero}: resumido")

por_pedazo = pd.DataFrame({numero: {"filas del resumen": len(r), "hogares": r["hogar"].nunique(),
                                    "kWh por hogar y día": r.loc[r["lecturas"] == 48, "kwh"].mean()}
                           for numero, r in resumenes.items()}).T
display(por_pedazo.round(2))
print(f"Leímos {remoto.bytes_bajados / 1e6:.0f} MB de los {remoto.tamano / 1e6:.0f} MB del ZIP")

Cada millón de filas quedó en unas veinte mil. Copien la tabla en la hoja y miren la última columna: el consumo promedio cambia de un pedazo a otro, porque cada pedazo trae hogares distintos. Quien lea un solo archivo «para tener una idea» se lleva la idea de treinta hogares.

## 4. La versión congelada y su huella

Para la clase no procesamos los 168 archivos. Yo procesé 24, repartidos a lo largo de la lista, con la misma función `resumir`, y guardé el resultado. Esa tabla está congelada: tiene un MD5, y si el archivo que llega es otro, la función se detiene.

La **huella** es un código corto que se calcula con el contenido de la tabla. Si cambia una celda, cambia la huella. Un resultado que no dice sobre qué huella se calculó no se puede reproducir.

In [ ]:
consumo = londres.congelada()

mio = resumenes[0].groupby("hogar")[["kwh", "lecturas"]].sum()
del_curso = consumo[consumo["hogar"].isin(mio.index)].groupby("hogar")[["kwh", "lecturas"]].sum()
print("¿Mi resumen del archivo 0 coincide con la tabla congelada?", bool(np.allclose(mio, del_curso.loc[mio.index], atol=1e-3)))

v1 = londres.versionar(consumo, "24 de 168 archivos; suma y cuenta por hogar y día; Null faltante; sin repetidas")
print("Huella de la versión 1:", v1)

## 5. Dos decisiones de limpieza

**Días incompletos.** Un día con 47 lecturas no es un día de bajo consumo. Es un día al que le falta media hora.

**Un panel que crece.** Los medidores se instalaron poco a poco. Miren las dos gráficas antes de leer la salida, y comparen con su apuesta A3.

In [ ]:
consumo["fecha"] = pd.to_datetime(consumo["fecha"])
print(f"Días con menos de 48 lecturas: {(consumo['lecturas'] < 48).mean() * 100:.1f} %")
completos = consumo[consumo["lecturas"] == 48].drop(columns="lecturas")
hogares_por_dia = completos.groupby("fecha")["hogar"].nunique()
total_por_dia = completos.groupby("fecha")["kwh"].sum()

fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 3.6))
(total_por_dia / 1000).plot(ax=izq, color="#D7263D"); izq.set(title="Consumo total de la muestra (MWh por día)", xlabel="")
hogares_por_dia.plot(ax=der, color="#1D4E89"); der.set(title="Hogares midiendo, por día", xlabel="")
plt.tight_layout(); plt.show()

enero = lambda serie, anio: serie[(serie.index.year == anio) & (serie.index.month == 1)].mean()
crecimiento = pd.DataFrame({anio: {"consumo total (kWh por día)": enero(total_por_dia, anio), "hogares midiendo": enero(hogares_por_dia, anio),
                                   "kWh por hogar y día": enero(total_por_dia / hogares_por_dia, anio)} for anio in (2012, 2013)})
crecimiento["veces"] = crecimiento[2013] / crecimiento[2012]
display(crecimiento.round(2))

El consumo total de la muestra se multiplicó entre un enero y el siguiente. Lo que creció fue el número de medidores. Por hogar, el consumo bajó. Quien sume kWh por día sin contar hogares le reporta a la empresa un crecimiento que no existió.

**Decisiones.** Solo entran los días con las 48 lecturas. La variable de trabajo es el consumo **promedio por hogar**. Y la serie empieza el primer día en que mide al menos la mitad del máximo de hogares. Cada decisión produce una versión nueva, con su huella y con la huella de su versión madre.

In [ ]:
primer_dia, ultimo_dia = londres.ventana(hogares_por_dia)
limpio = completos[completos["fecha"].between(primer_dia, ultimo_dia)].reset_index(drop=True)
v2 = londres.versionar(limpio.assign(fecha=limpio["fecha"].dt.strftime("%Y-%m-%d")),
                       f"solo días con 48 lecturas; del {primer_dia.date()} al {ultimo_dia.date()}", madre=v1)
serie = londres.serie_por_hogar(limpio)
print(f"Serie: {len(serie)} días, del {serie.index.min().date()} al {serie.index.max().date()}")
print(f"Consumo promedio: {serie.mean():.2f} kWh por hogar y día | factura diaria: {serie.mean() * TARIFA_PLANA / 100:.2f} libras")
display(londres.registro()[["huella", "filas", "regla", "madre"]])

## 6. Conocer el dato

Tres miradas: cómo se reparte el consumo entre hogares, cómo cambia con los meses y cómo cambia con el día de la semana.

In [ ]:
por_hogar = limpio.groupby("hogar")["kwh"].mean()
por_mes = serie.groupby(serie.index.month).mean()
por_dia = serie.groupby(serie.index.dayofweek).mean()

fig, ejes = plt.subplots(1, 3, figsize=(15, 3.8))
ejes[0].hist(por_hogar.clip(upper=40), bins=40, color="#1D4E89")
ejes[0].axvline(por_hogar.mean(), color="#D7263D"); ejes[0].axvline(por_hogar.median(), color="#D7263D", ls=":")
ejes[0].set(title="Consumo por hogar (kWh por día)", xlabel="línea continua: media; punteada: mediana")
ejes[1].bar(["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"], por_mes, color="#6C4AB6")
ejes[1].set(title="Consumo por mes")
ejes[2].bar(["lun", "mar", "mié", "jue", "vie", "sáb", "dom"], por_dia, color="#1B998B"); ejes[2].set(title="Consumo por día de la semana", ylim=(por_dia.min() * 0.9, por_dia.max() * 1.02))
plt.tight_layout(); plt.show()
print(f"Media por hogar: {por_hogar.mean():.2f} kWh | mediana: {por_hogar.median():.2f} kWh")
print(f"El 10 % de hogares que más consume gasta el {por_hogar.nlargest(len(por_hogar) // 10).sum() / por_hogar.sum() * 100:.0f} % de la energía")
print(f"Mes más alto sobre mes más bajo: {por_mes.max() / por_mes.min():.2f} veces")

**Lectura.** La media queda por encima de la mediana, porque pocos hogares consumen mucho. La estación domina la serie: en invierno se consume bastante más que en verano. El domingo es el día más alto de la semana, y la diferencia con los demás días es pequeña.

## 7. Predecir: ¿cuánto consume mañana un hogar promedio?

Antes de cualquier modelo hay dos pronósticos que no cuestan nada:

$$\hat C_{t+1}=C_t\quad\text{(igual que hoy)}\qquad\text{y}\qquad\hat C_{t+1}=C_{t-6}\quad\text{(igual que hace una semana)}.$$

El error se mide con el MAPE, el promedio de $|\hat C_{t+1}-C_{t+1}|/C_{t+1}$, en porcentaje.

Contra esas dos reglas compiten tres modelos. Usan el consumo reciente, el día de la semana y la época del año. El tiempo se parte en tres tramos: entrenamiento hasta junio de 2013, validación de julio a octubre para escoger, y prueba de noviembre de 2013 a febrero de 2014 para reportar. Comparen con su apuesta A4.

In [ ]:
errores, pronosticos = londres.competir(serie)
print("Días en cada tramo:", errores.attrs["dias"])
display(errores.sort_values("validación").round(2))

mejor = errores["validación"].idxmin()
elegido = mejor if errores.loc[mejor, "prueba"] < errores.loc["igual que hoy", "prueba"] else "igual que hoy"
print(f"Ganador en validación: {mejor} | se usa: {elegido}")
print(f"MAPE en prueba: {errores.loc[elegido, 'prueba']:.2f} % | igual que hoy: {errores.loc['igual que hoy', 'prueba']:.2f} %")

pronosticos[["real", "igual que hoy", elegido]].plot(figsize=(11, 3.6), color=["black", "#9AA5B1", "#D7263D"], style=["-", "-", "--"])
plt.title("La prueba: el consumo real y dos pronósticos"); plt.ylabel("kWh por hogar"); plt.xlabel(""); plt.show()

La serie es dócil. Promediar cientos de hogares cancela los caprichos de cada uno, y por eso «igual que hoy» ya se equivoca poco. El modelo le quita cerca de un punto de error. En la clase 1, con el precio de la luz, ningún modelo le ganó al pronóstico ingenuo.

### El mismo cálculo sobre otra versión del dato

Repito la línea base sobre la versión 1, la que no tenía ninguna de las dos decisiones de limpieza.

In [ ]:
serie_v1 = londres.serie_por_hogar(consumo)
ingenuo = lambda s: londres.mape(s[1:], s.shift(1)[1:])
comparacion = pd.DataFrame({
    f"versión {v1} (sin limpiar)": {"días": len(serie_v1), "kWh por hogar y día": serie_v1.mean(), "MAPE de igual que hoy (%)": ingenuo(serie_v1)},
    f"versión {v2} (limpia)": {"días": len(serie), "kWh por hogar y día": serie.mean(), "MAPE de igual que hoy (%)": ingenuo(serie)},
}).T
display(comparacion.round(2))

Los dos renglones salen del mismo código y de la misma fuente. Solo cambió la regla de limpieza. Si dos personas reportan números distintos y ninguna anotó la huella, no hay manera de saber si la diferencia viene del modelo o del dato.

## 8. Decidir: ¿cuándo se manda el aviso?

Llamo **día alto** a un día cuyo consumo supera el nivel $U$, el percentil 75 del consumo anterior a la prueba. La empresa avisa la víspera si el pronóstico supera un umbral. Cada renglón de la tabla es una política.

In [ ]:
U = serie[:"2013-10-31"].quantile(0.75)
print(f"Nivel de día alto: {U:.2f} kWh por hogar | días altos en la prueba: {int((pronosticos['real'] > U).sum())} de {len(pronosticos)}")
decision = londres.avisos(pronosticos["real"], pronosticos[elegido], U)
display(decision.round(2))

Al bajar el umbral se atrapan más días altos y se mandan más avisos en falso. Un aviso en falso gasta la paciencia del cliente. Un día alto sin aviso le cuesta a la empresa energía comprada a última hora. La tabla muestra cuánto se cambia de un error por el otro, y la empresa escoge según cuál le duele más.

**En la hoja:** escriban la decisión en tres frases, para una gerente comercial que no sabe qué es un percentil.

## 9. El post

Tres láminas con lo que dijo el dato. Las cifras salen de las celdas de arriba.

In [ ]:
from taller import carrusel as c
c.preparar_fuentes()
PIE = "Fuente: UK Power Networks, medidores inteligentes de Londres (London Datastore, CC BY 4.0).\nCálculos propios sobre 24 de 168 archivos. Carlos Manuel Orrego Franco"
veces = crecimiento["veces"]
coma = lambda numero: f"{numero:.1f}".replace(".", ",")
c.lamina_barras({"Consumo total\nde la muestra": veces.iloc[0], "Hogares\nmidiendo": veces.iloc[1], "Consumo\npor hogar": veces.iloc[2]},
                {"Consumo total\nde la muestra": c.ROJO, "Hogares\nmidiendo": c.AZUL, "Consumo\npor hogar": c.MORADO}, "LONDRES, 2012 A 2013",
                f"El consumo se multiplicó por {coma(veces.iloc[0])}.\nLos medidores, por {coma(veces.iloc[1])}",
                "Enero de 2013 dividido por enero de 2012. Lo que creció\nfue la muestra: por hogar, el consumo bajó.", PIE, "2/4", "londres_2.png", sufijo=" veces")
plt.show()

In [ ]:
pedazos = por_pedazo["kWh por hogar y día"]
c.lamina_barras({f"Archivo {n}\n{int(por_pedazo.loc[n, 'hogares'])} hogares": v for n, v in pedazos.items()}, {f"Archivo {n}\n{int(por_pedazo.loc[n, 'hogares'])} hogares": c.AZUL for n in pedazos.index},
                "UN MILLÓN DE FILAS", f"Un millón de filas son\n{int(por_pedazo['hogares'].min())} a {int(por_pedazo['hogares'].max())} hogares",
                f"Consumo promedio por hogar y día en {len(pedazos)} archivos de un millón de filas.\nEl archivo está ordenado por hogar: cada pedazo es otra muestra.", PIE, "3/4", "londres_3.png", sufijo=" kWh")
plt.show()

In [ ]:
nombres = {"igual que hoy": "Igual\nque hoy", "igual que hace una semana": "Igual que hace\nuna semana", elegido: elegido.capitalize().replace(" ", "\n")}
barras = {nombres[k]: errores.loc[k, "prueba"] for k in nombres}
c.lamina_barras(barras, {k: (c.ROJO if k == nombres[elegido] else c.GRIS) for k in barras}, "EL CONSUMO DE MAÑANA",
                f"“Igual que hoy” se equivoca {coma(errores.loc['igual que hoy', 'prueba'])} %.\nEl modelo, {coma(errores.loc[elegido, 'prueba'])} %",
                "Error porcentual medio al pronosticar el consumo del día siguiente,\nen cuatro meses que el modelo no vio.", PIE, "4/4", "londres_4.png")
plt.show()

## Bitácora

| Decisión | Alternativa | Por qué |
|---|---|---|
| Leer el ZIP por rangos y resumir cada pedazo | Bajar los 8,5 GB y leerlos de un golpe | No cabe cómodo en memoria, y la empresa necesita días, no medias horas |
| Guardar sumas y cuentas | Guardar promedios por pedazo | Un hogar puede quedar partido entre dos archivos |
| 24 archivos repartidos por toda la lista | Los primeros 24 | El archivo está ordenado por hogar: el principio es una muestra de conveniencia |
| Solo días con 48 lecturas | Usar todos los días | Un día incompleto parece un día de bajo consumo |
| Consumo promedio por hogar | Consumo total | El número de medidores creció durante 2012 |
| Empezar cuando mide la mitad del máximo de hogares | Empezar en el primer día | Con pocos hogares el promedio salta |
| Escoger en validación y reportar en prueba | Escoger en prueba | Escoger en prueba sería mirar el examen |

## Cambien una línea

Vuelvan a la celda donde dice `ARCHIVOS = 4`, pongan `ARCHIVOS = 8` y corran todo de nuevo. Llenen la tabla de la hoja. Fíjense en qué cambia y en qué no: la tabla por pedazo cambia, y las huellas y los errores del pronóstico se quedan iguales, porque salen de la versión congelada.

In [ ]:
print("Fuente: UK Power Networks, SmartMeter Energy Consumption Data in London Households, London Datastore, CC BY 4.0")
print(f"Archivos leídos en vivo: {list(resumenes)} | versión congelada: {v1} | versión limpia: {v2}")